# 02 Privacy first: every row is a neighbour — notebook

Before Maya shares anything with Doc Mon at the PESU, she must turn Jun's raw export into
a file that protects the people in it. In this notebook you will:

1. sort the columns into **direct identifiers**, **indirect identifiers** and
   **sensitive health information**;
2. drop direct identifiers — and catch Tala **leaving a name behind** in the remarks;
3. see why a plain **hash of a name is not anonymous**, and make a **salted** pseudonymous ID;
4. **generalise** age and dates;
5. **suppress small cells** in a table you would publish;
6. measure **re-identification risk** by counting how many people are unique.

```{warning}
**The names, phone numbers and addresses in `santa_luz_linelist_raw.csv` are FAKE.**
They were invented by a computer and deliberately *look* real so you can practise.
Santa Luz City and everyone in it are fictional. Real line lists never leave your office's
protected computers — and never go into a notebook you share or an AI chat.
```

This is good practice for training, not legal advice. In real work, follow your
office's Data Protection Officer and the DOH rules that apply to you.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

## 1. Open Jun's raw export

The first line of the file is a note saying the data are synthetic, so we skip it with
`skiprows=1`. `dtype=str` keeps every cell as text exactly as typed, and
`keep_default_na=False` keeps blank cells as empty text `""` instead of turning them
into "missing".

In [ ]:
import pandas as pd

raw = pd.read_csv(
    "data/synthetic/ph/santa_luz_linelist_raw.csv",
    skiprows=1, dtype=str, keep_default_na=False,
)
print(raw.shape)
raw.head(3)      # FAKE names, phones and addresses

## 2. Which columns can identify a person?

- A **direct identifier** points to one person on its own: name, phone number, street
  address, PhilHealth or other ID numbers.
- An **indirect identifier** (a *quasi-identifier*) does not name anyone alone, but a few
  of them together can: barangay + age + sex + date of onset may fit only one person.
- **Sensitive health information** is what the row is *about*: the disease, whether the
  person was admitted, whether they died.
- **Free text** (`Remarks`) can hold any of the above. Treat it as a direct identifier
  until you have read every entry.

Writing the roles down as a dictionary has a bonus: we can **check** that every column
got a role, so nothing slips through unclassified.

In [ ]:
roles = {
    "Patient Name": "direct identifier",
    "Contact No.": "direct identifier",
    "Address": "direct identifier",
    "Remarks": "free text (treat as direct)",
    "Reporting Unit": "indirect identifier",
    "Barangay": "indirect identifier",
    "Age": "indirect identifier",
    "Sex": "indirect identifier",
    "Date Reported": "indirect identifier",
    "Date of Onset": "indirect identifier",
    "Date Consulted": "indirect identifier",
    "Date Lab Confirmed": "indirect identifier",
    "Disease": "sensitive health information",
    "Classification": "sensitive health information",
    "Admitted": "sensitive health information",
    "Outcome": "sensitive health information",
    "Dengue Severity": "sensitive health information",
    "Waded in Floodwater": "sensitive health information",
    "Did Cleanup Work": "sensitive health information",
}

# Check: every column has a role, and no role is given to a column that does not exist
assert set(roles) == set(raw.columns), set(raw.columns) ^ set(roles)
print(pd.Series(roles).value_counts())

## 3. Ask Tala — with the schema, never the rows

Maya does **not** paste any rows. She describes the columns only:

> *I have a pandas DataFrame `raw` with 19 text columns: Date Reported, Reporting Unit,
> Patient Name, Sex, Age, Contact No., Address, Barangay, Disease, Date of Onset,
> Date Consulted, Classification, Date Lab Confirmed, Admitted, Outcome,
> Waded in Floodwater, Did Cleanup Work, Dengue Severity, Remarks.
> I cannot share any rows: they are confidential patient data.
> Write pandas code that removes the direct identifiers so I can share the table
> with our provincial epidemiologist.*

Tala answered:

In [ ]:
# Tala's code
direct_identifiers = ["Patient Name", "Contact No.", "Address"]
shared = raw.drop(columns=direct_identifiers)

print(shared.shape)
print(list(shared.columns))

## 4. Check it: is any name still in the file?

Tala's code ran without an error, and the three columns are gone. But "no error" is
not the same as "no names". A simple test: take every **first name** from the column we
just dropped, and search **every remaining cell** for it. The search pattern
`\b(Ana|Dennis|...)\b` means "one of these names as a whole word".

In [ ]:
first_names = sorted(set(raw["Patient Name"].str.split().str[0]))
pattern = r"\b(?:" + "|".join(first_names) + r")\b"

hits = shared.apply(lambda column: column.str.contains(pattern)).sum()
print("Cells containing a first name, by column:")
print(hits[hits > 0])

shared.loc[shared["Remarks"].str.contains(pattern), "Remarks"].head(5)

### Tala got it wrong

Forty remarks still name a person — *"anak/kapatid ni Dennis (kapitbahay)"*: the child
or sibling of Dennis, a neighbour. That remark names **someone else**, and together
with the barangay it can point straight at a household. Other remarks are risky too:
*"estudyante, Grade 5"* narrows a child down to one class.

Tala removed the columns *called* name, phone and address. It could not see inside
`Remarks`, because Maya (rightly) never showed it any rows. **The model only knows what
you tell it; checking the output is your job.**

The fix: drop `Remarks` too (anything useful in it should be coded into a yes/no column
inside the office first), and drop `Reporting Unit` — some units sent only one or two
reports, so the unit name can point to a person. Then run the same check again: it must
find **zero**.

In [ ]:
remove = ["Patient Name", "Contact No.", "Address", "Remarks", "Reporting Unit"]
shared = raw.drop(columns=remove)

hits = shared.apply(lambda column: column.str.contains(pattern)).sum()
assert hits.sum() == 0, "A name is still in the shared table!"
print(f"No first names left in {shared.shape[0]} rows x {shared.shape[1]} columns.")
print("Reports per unit (why the unit name was risky):")
print(raw["Reporting Unit"].value_counts().tail(3))

## 5. A pseudonymous ID: why a plain hash is not enough

We still need an ID so that Doc Mon can say *"check case 17 again"* and Maya can find it.
A **hash** turns text into a fixed jumble of characters; the same text always gives the
same jumble. Many people think hashing a name makes it anonymous. It does not: anyone
with a **list of likely names** (a voters' list, a class list, social media) can hash
every name on it and look for a match. The cell below plays that "attacker" with a
made-up list of five names.

In [ ]:
import hashlib


def plain_hash(name: str) -> str:
    return hashlib.sha256(name.encode()).hexdigest()[:12]


leaked_id = plain_hash("Juan P. Santos")          # what an unsalted file would contain
print("ID in the shared file:", leaked_id)

guess_list = ["Maria L. Reyes", "Jose A. Cruz", "Juan P. Santos", "Ana B. Garcia", "Mark D. Lim"]
for guess in guess_list:
    if plain_hash(guess) == leaked_id:
        print("Attacker found the person:", guess)

**Add a secret salt.** We mix a long random secret (the *salt*) into the hash with
`hmac`. Without the salt, nobody can redo the hashing, so the guess list is useless. The
office keeps the salt in a locked file **inside the office** — never in the notebook,
never in an AI chat, never sent with the data. Keeping the same salt from week to week
means the same person keeps the same ID across exports.

Before hashing we **normalise** the name (capital letters, no dots, single spaces) so
`Juan P. Santos` and `JUAN P SANTOS` get the same ID. The checks at the end test exactly
that, and that a different salt gives a different ID.

`secrets.token_hex(16)` makes a new random salt every time it runs, so **your IDs will
differ from anyone else's** — that is the point.

In [ ]:
import hmac
import secrets

SALT = secrets.token_hex(16)     # demo only: in real work, create once and lock it away


def pseudo_id(name: str, salt: str) -> str:
    key = " ".join(name.upper().replace(".", " ").split())   # normalise the name
    digest = hmac.new(salt.encode(), key.encode(), hashlib.sha256).hexdigest()
    return "P-" + digest[:10]


print(pseudo_id("Juan P. Santos", SALT))

# Checks
assert pseudo_id("Juan P. Santos", SALT) == pseudo_id("JUAN P SANTOS ", SALT)
assert pseudo_id("Juan P. Santos", SALT) != pseudo_id("Juan P. Santos", "another salt")
assert all(pseudo_id(g, SALT) != leaked_id for g in guess_list)

shared.insert(0, "pseudo_id", raw["Patient Name"].map(lambda n: pseudo_id(n, SALT)))
shared.head(3)

## 6. Generalise: age groups and weeks

Exact age and exact dates are strong indirect identifiers. For a table that leaves the
office, we **coarsen** them: age → age group, date → week.

The raw ages and dates are still messy (`8 mos`, `24-Jul-26`, `46227`); chapter 03 cleans
them. So from here on we use **`santa_luz_linelist.csv`**, the cleaned and de-identified
file that chapter 03 produces (sequential IDs `SLC-0001`…, no names, phones, addresses,
reporting units or remarks).

`pd.cut` sorts ages into bins. `right=False` means each bin includes its left edge, so
5 goes into `5-14`, not `0-4`. The **check**: our groups must match the file's own
`age_group` column for every row with an age.

In [ ]:
df = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv")

labels = ["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]
df["my_age_group"] = pd.cut(df["age"], bins=[0, 5, 15, 25, 45, 65, 200],
                            right=False, labels=labels)

has_age = df["age"].notna()
assert (df.loc[has_age, "my_age_group"].astype(str) == df.loc[has_age, "age_group"]).all()
print(f"Age groups match for all {has_age.sum()} cases with an age "
      f"({(~has_age).sum()} ages are missing).")

# Dates -> ISO week of onset (Monday to Sunday)
df["onset_week"] = pd.to_datetime(df["date_onset"]).dt.isocalendar().week
print(df[["date_onset", "onset_week"]].head(3))

## 7. Small-cell suppression before you publish

A published table with a **1** in it can be a person. *"One dengue death in Buhangin"*
is news the whole barangay can put a name to. The common rule is to hide counts from
**1 to 4** and show `<5` instead (zero stays zero: it reveals no one).

We try it on the most sensitive table: deaths by barangay and disease. One more trap:
if a column has **only one** hidden cell, anyone who knows the column total (and city
totals are usually public) can subtract the visible cells and recover it. So the cell
also **checks** every column: one hidden cell means that column cannot be published by
barangay. For such a column we publish only the city total and leave the barangay out.
Even a city total of 1 is sensitive: whether to publish it is a decision for Doc Mon and
the office's Data Protection Officer, not for the code.

In [ ]:
def suppress(table: pd.DataFrame, threshold: int = 5) -> pd.DataFrame:
    if threshold < 1:
        raise ValueError("threshold must be at least 1")
    small = (table > 0) & (table < threshold)
    return table.astype(object).mask(small, f"<{threshold}")


deaths = df[df["outcome"] == "died"]
death_table = pd.crosstab(deaths["barangay"], deaths["disease"])
print(f"Deaths: {len(deaths)} in {len(death_table)} barangays")

safe_table = suppress(death_table)
print(safe_table)

# Check: a column with exactly ONE hidden cell can be recovered from its total
hidden = (safe_table == "<5").sum()
recoverable = list(hidden[hidden == 1].index)
print("Columns with a single hidden cell:", recoverable)

publishable = safe_table.drop(columns=recoverable)
assert ((publishable == "<5").sum() != 1).all()
print(publishable)
for column in recoverable:
    print(f"{column}: publish the city total only ({death_table[column].sum()}), no barangay.")

## 8. Re-identification check: how many people are unique?

A row is risky if it is **unique** on its indirect identifiers — nobody else in the file
shares its barangay, age, sex, disease, onset date… The idea behind **k-anonymity**: every
combination should be shared by at least *k* people (say *k* = 5), so any row could be
one of at least five.

The function counts, for each row, how many rows share its combination (*k*), and
reports how many rows are unique (*k* = 1). We try four levels of detail, from exact to
coarse. Rows with a missing value in the chosen columns are left out of each count.

In [ ]:
import matplotlib.pyplot as plt


def uniqueness(data: pd.DataFrame, quasi: list[str]) -> dict:
    rows = data.dropna(subset=quasi)
    k = rows.groupby(quasi)[quasi[0]].transform("size")   # group size for each row
    return {"rows": len(rows), "unique (k=1)": int((k == 1).sum()), "k<5": int((k < 5).sum())}


levels = {
    "barangay + age + sex + disease + onset date": ["barangay", "age", "sex", "disease", "date_onset"],
    "barangay + age + sex + disease": ["barangay", "age", "sex", "disease"],
    "barangay + age group + sex + disease": ["barangay", "age_group", "sex", "disease"],
    "band + age group + sex + disease": ["band", "age_group", "sex", "disease"],
}
risk = pd.DataFrame({name: uniqueness(df, cols) for name, cols in levels.items()}).T
risk["% unique"] = (risk["unique (k=1)"] / risk["rows"] * 100).round(1)
print(risk)

ax = risk["% unique"].iloc[::-1].plot.barh(color="#CE1126", figsize=(8, 3.2))
ax.set_xlabel("% of cases unique on these columns (k = 1)")
ax.set_title("Coarser columns, fewer unique people (Santa Luz, fictional)")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

## Your turn

1. **More columns, more risk.** Add `"case_class"` to `my_columns` below and run the cell.
   Does the number of unique rows go up or down? *Hint:* every extra column can only
   split groups into smaller ones.
2. **A stricter threshold.** Some offices suppress counts under 10. Run
   `suppress(death_table, threshold=10)` (already in the cell) and read the result: how
   much of the table is still useful?
3. **Write a safe prompt.** In a text cell, write a prompt asking your AI assistant for
   code that counts dengue cases by `barangay` and `age_group` and suppresses cells
   under 5. Use only column names and two or three obviously made-up rows (e.g.
   barangay `"Barangay X"`). Then compare its code with `suppress()` above.
   *Hint:* check that its code leaves zeros alone.

In [ ]:
# --- Your turn 1 ---
my_columns = ["band", "age_group", "sex", "disease"]      # <- add "case_class"
print(uniqueness(df, my_columns))

# --- Your turn 2 ---
suppress(death_table, threshold=10)

## Baon

- **Look inside every column.** Dropping the columns *called* name, phone and address is
  not enough: free-text remarks named 40 other people. Search the result for what you
  removed.
- **Pseudonymise with a secret.** A plain hash of a name can be reversed with a guess
  list; a salted ID (salt locked in the office) or a sequential ID with a locked key
  table cannot.
- **Coarsen, suppress, then measure.** Age groups, weeks and `<5` cells cut the share of
  unique people sharply — and the uniqueness check tells you whether you have done enough.

Next: [chapter 03, *From messy line list to clean data*](../03_clean_linelist.md).